# 05 - The gold layer, Jinja and macros

**You will learn**: dimensional modelling (star schema), Jinja, macros, packages (`dbt_utils`).

**You will build**: the dimensions, the sales fact table `fct_sales`, the `net_amount` macro and business marts.

## The star schema

```
                 dim_date
                    │
 dim_customer ── fct_sales ── dim_product
                    │
      dim_store ────┴──── dim_sales_person
```

* **Fact** table: one row per *event*, here **one row per order line**, with numeric measures (quantity, amounts).
* **Dimension** tables: one row per *thing* (product, store, customer...) with descriptive columns.
* **Marts**: pre-aggregated tables answering a specific business question.

In [ ]:
from helpers import *

## 1. Jinja in 5 minutes

dbt models are SQL files **processed by Jinja** (a Python templating language) before being sent to the warehouse.

| Syntax | Purpose | Example |
|---|---|---|
| `{{ ... }}` | print an expression | `{{ ref('stg_products') }}` |
| `{% ... %}` | statement (if, for, set) | `{% if is_incremental() %} ... {% endif %}` |
| `{# ... #}` | comment | |

Try it: `dbt compile --inline` renders a snippet without touching the warehouse.

In [ ]:
dbt("compile --inline \"select {{ 1 + 1 }} as two, '{{ target.name }}' as target_name, {{ ref('stg_products') }} as product_table\"")

Jinja can loop too:

In [ ]:
dbt("compile --inline \"{% for c in ['category', 'brand'] %}select '{{ c }}' as dimension_name {% if not loop.last %}union all {% endif %}{% endfor %}\"")

## 2. Your first macro: `net_amount`

A **macro** is a reusable Jinja function, stored in `macros/`. We compute the net amount of an order line in several places,
so we write the formula once:

> net amount = quantity x unit price x (1 - discount in percent / 100)

**Exercise A.** Complete the macro (the file is created by the next cell).

In [ ]:
%%writefile ../project/macros/net_amount.sql
{#
  Net amount of an order line: quantity * unit price, minus the discount (in percent).
  Usage: {{ net_amount('quantity', 'unit_price', 'discount_pct') }}
#}
{% macro net_amount(quantity, unit_price, discount_pct, precision=2) -%}
    {# TODO: return round(<quantity> * <unit_price> * (1 - <discount_pct> / 100.0), <precision>) #}
    0
{%- endmacro %}


Test it: 2 units at CHF 10 with 20% discount must give 16.

In [ ]:
dbt("compile --inline \"select {{ net_amount('2', '10', '20') }} as net\"")

## 3. Packages

`dbt_utils` (installed in notebook 01) provides tests and macros. We use its `date_spine` macro to generate a calendar
dimension without any source data. The model is provided, read it:

In [ ]:
%%writefile ../project/models/gold/dim_date.sql
-- Calendar dimension generated with the dbt_utils.date_spine macro.
with spine as (

    {{ dbt_utils.date_spine(
        datepart="day",
        start_date="cast('2024-01-01' as date)",
        end_date="cast('2029-01-01' as date)"
    ) }}

)

select
    cast(date_day as date) as date_day,
    year(date_day) as year,
    quarter(date_day) as quarter,
    month(date_day) as month,
    date_format(date_day, 'MMMM') as month_name,
    weekofyear(date_day) as week_of_year,
    weekday(date_day) + 1 as day_of_week,  -- 1 = Monday
    date_format(date_day, 'EEEE') as day_name,
    weekday(date_day) >= 5 as is_weekend,
    case
        when month(date_day) in (12, 1, 2) then 'winter'
        when month(date_day) in (3, 4, 5) then 'spring'
        when month(date_day) in (6, 7, 8) then 'summer'
        else 'autumn'
    end as season
from spine


## 4. Dimensions

**Exercise B.** `dim_product`: the product catalog with an extra column `unit_margin` (`list_price - unit_cost`, rounded to 2 decimals).
`dim_store` and `dim_sales_person` are provided (the latter joins to the stores to show the store name).
`dim_customer` was written in notebook 03.

In [ ]:
%%writefile ../project/models/gold/dim_product.sql
select
    product_id,
    product_name,
    category,
    brand,
    list_price,
    unit_cost,
    -- TODO: unit_margin = list_price - unit_cost, rounded to 2 decimals
    is_active
from {{ ref('stg_products') }}


In [ ]:
%%writefile ../project/models/gold/dim_store.sql
select
    store_id,
    store_name,
    city,
    canton,
    store_type,
    is_online,
    opened_date
from {{ ref('stg_stores') }}


In [ ]:
%%writefile ../project/models/gold/dim_sales_person.sql
select
    sp.sales_person_id,
    sp.full_name,
    sp.email,
    sp.hire_date,
    sp.store_id,
    s.store_name,
    s.canton as store_canton
from {{ ref('stg_sales_persons') }} as sp
left join {{ ref('stg_stores') }} as s
    on sp.store_id = s.store_id


## 5. The fact table

**Exercise C.** `fct_sales`, grain: **one row per order line**. Join lines to orders (inner) and to products (left, for the cost),
then compute the amounts. Use your macro for `net_amount`.

* `gross_amount` = quantity x unit_price
* `discount_amount` = gross - net
* `cost_amount` = quantity x product unit_cost
* `margin_amount` = net - cost

In [ ]:
%%writefile ../project/models/gold/fct_sales.sql
-- Grain: one row per order line.
with orders as (

    select * from {{ ref('stg_sales_orders') }}

),

lines as (

    select * from {{ ref('stg_sales_order_lines') }}

),

products as (

    select product_id, unit_cost from {{ ref('stg_products') }}

)

select
    l.order_line_id,
    l.order_id,
    o.order_date,
    o.order_day,
    o.customer_id,
    o.store_id,
    o.sales_person_id,
    l.product_id,
    o.channel,
    o.payment_method,
    o.status as order_status,
    l.quantity,
    l.unit_price,
    l.discount_pct,
    round(l.quantity * l.unit_price, 2) as gross_amount,
    -- TODO: discount_amount = gross amount - net amount (use the macro net_amount)
    0 as discount_amount,
    -- TODO: net_amount using {{ net_amount('l.quantity', 'l.unit_price', 'l.discount_pct') }}
    0 as net_amount,
    round(l.quantity * p.unit_cost, 2) as cost_amount,
    -- TODO: margin_amount = net amount - cost_amount
    0 as margin_amount,
    o.updated_at as order_updated_at
from lines as l
inner join orders as o
    on l.order_id = o.order_id
left join products as p
    on l.product_id = p.product_id


## 6. Marts

A mart aggregates the fact table for a business question. Only **completed** orders count as revenue.

**Exercise D.** `mart_revenue_by_category_month`: per month and product category, the number of orders, units sold, net revenue and margin.
The other marts are provided.

In [ ]:
%%writefile ../project/models/gold/mart_revenue_by_category_month.sql
-- Monthly revenue per product category (completed orders only).
select
    date_trunc('month', f.order_day) as month,
    p.category,
    -- TODO: count(distinct f.order_id) as orders
    -- TODO: sum(f.quantity) as units_sold
    -- TODO: sum(f.net_amount) as net_revenue
    -- TODO: sum(f.margin_amount) as margin
    1 as placeholder
from {{ ref('fct_sales') }} as f
inner join {{ ref('dim_product') }} as p
    on f.product_id = p.product_id
where f.order_status = 'completed'
group by 1, 2


In [ ]:
%%writefile ../project/models/gold/mart_store_performance.sql
-- Monthly performance per store (completed orders only).
select
    date_trunc('month', f.order_day) as month,
    s.store_id,
    s.store_name,
    s.store_type,
    count(distinct f.order_id) as orders,
    sum(f.net_amount) as net_revenue,
    round(sum(f.net_amount) / count(distinct f.order_id), 2) as avg_basket,
    sum(f.margin_amount) as margin
from {{ ref('fct_sales') }} as f
inner join {{ ref('dim_store') }} as s
    on f.store_id = s.store_id
where f.order_status = 'completed'
group by 1, 2, 3, 4


In [ ]:
%%writefile ../project/models/gold/mart_sales_person_ranking.sql
-- Sales persons ranked by revenue, with the cumulative share of total revenue (the 80/20 view).
with revenue as (

    select
        sp.sales_person_id,
        sp.full_name,
        sp.store_name,
        count(distinct f.order_id) as orders,
        sum(f.net_amount) as net_revenue
    from {{ ref('fct_sales') }} as f
    inner join {{ ref('dim_sales_person') }} as sp
        on f.sales_person_id = sp.sales_person_id
    where f.order_status = 'completed'
    group by 1, 2, 3

)

select
    rank() over (order by net_revenue desc) as revenue_rank,
    sales_person_id,
    full_name,
    store_name,
    orders,
    net_revenue,
    round(net_revenue / sum(net_revenue) over (), 4) as revenue_share,
    round(sum(net_revenue) over (order by net_revenue desc) / sum(net_revenue) over (), 4) as cumulative_revenue_share
from revenue


In [ ]:
%%writefile ../project/models/gold/mart_revenue_by_country.sql
-- Revenue per customer country and continent (uses the `countries` seed through dim_customer).
select
    c.country_code,
    c.country_name,
    c.continent,
    c.is_eu,
    count(distinct f.customer_id) as customers,
    count(distinct f.order_id) as orders,
    sum(f.net_amount) as net_revenue
from {{ ref('fct_sales') }} as f
inner join {{ ref('dim_customer') }} as c
    on f.customer_id = c.customer_id
where f.order_status = 'completed'
group by 1, 2, 3, 4


## 7. Tests for gold and build

The tests for the gold layer are provided: primary keys, and `relationships` from the fact to every dimension (referential integrity).
Two singular tests reconcile the fact table with silver.

In [ ]:
%%writefile ../project/models/gold/_gold.yml
version: 2
models:
- name: dim_date
  columns:
  - name: date_day
    data_tests:
    - unique
    - not_null
  - name: season
    data_tests:
    - accepted_values:
        arguments:
          values:
          - winter
          - spring
          - summer
          - autumn
- name: dim_product
  columns:
  - name: product_id
    data_tests:
    - unique
    - not_null
  - name: unit_margin
- name: dim_store
  columns:
  - name: store_id
    data_tests:
    - unique
    - not_null
- name: dim_sales_person
  columns:
  - name: sales_person_id
    data_tests:
    - unique
    - not_null
- name: dim_customer
  columns:
  - name: customer_id
    data_tests:
    - unique
    - not_null
  - name: country_name
    data_tests:
    - not_null
  - name: loyalty_tier
- name: fct_sales
  columns:
  - name: order_line_id
    data_tests:
    - unique
    - not_null
  - name: order_id
    data_tests:
    - not_null
  - name: order_day
    data_tests:
    - relationships:
        arguments:
          to: ref('dim_date')
          field: date_day
  - name: customer_id
    data_tests:
    - relationships:
        arguments:
          to: ref('dim_customer')
          field: customer_id
  - name: store_id
    data_tests:
    - not_null
    - relationships:
        arguments:
          to: ref('dim_store')
          field: store_id
  - name: sales_person_id
    data_tests:
    - relationships:
        arguments:
          to: ref('dim_sales_person')
          field: sales_person_id
  - name: product_id
    data_tests:
    - not_null
    - relationships:
        arguments:
          to: ref('dim_product')
          field: product_id
  - name: order_status
    data_tests:
    - accepted_values:
        arguments:
          values:
          - completed
          - cancelled
          - returned
  - name: gross_amount
  - name: discount_amount
  - name: net_amount
    data_tests:
    - dbt_utils.accepted_range:
        arguments:
          min_value: 0
  - name: margin_amount
  - name: order_updated_at
- name: mart_revenue_by_category_month
  data_tests:
  - dbt_utils.unique_combination_of_columns:
      arguments:
        combination_of_columns:
        - month
        - category
  columns:
  - name: net_revenue
- name: mart_store_performance
  data_tests:
  - dbt_utils.unique_combination_of_columns:
      arguments:
        combination_of_columns:
        - month
        - store_id
- name: mart_sales_person_ranking
  columns:
  - name: sales_person_id
    data_tests:
    - unique
    - not_null
  - name: cumulative_revenue_share
- name: mart_revenue_by_country
  columns:
  - name: country_code
    data_tests:
    - unique
    - not_null


In [ ]:
%%writefile ../project/tests/assert_fct_sales_reconciles_with_silver.sql
-- Singular test: returns a row (= fails) if the total net amount in the gold fact table
-- differs from the total recomputed directly from the silver tables.
with gold as (

    select sum(net_amount) as net_amount from {{ ref('fct_sales') }}

),

silver as (

    select sum({{ net_amount('l.quantity', 'l.unit_price', 'l.discount_pct') }}) as net_amount
    from {{ ref('stg_sales_order_lines') }} as l
    inner join {{ ref('stg_sales_orders') }} as o
        on l.order_id = o.order_id

)

select
    gold.net_amount as gold_net_amount,
    silver.net_amount as silver_net_amount
from gold
cross join silver
where abs(gold.net_amount - silver.net_amount) > 0.01


In [ ]:
%%writefile ../project/tests/assert_every_order_has_lines.sql
{{ config(severity='warn') }}

-- Singular test: every silver order should have at least one line in the fact table.
-- Expected to warn: an order whose only line was dirty in bronze (negative quantity,
-- unknown product) loses that line in silver and is left without any line.
select o.order_id
from {{ ref('stg_sales_orders') }} as o
left join {{ ref('fct_sales') }} as f
    on o.order_id = f.order_id
where f.order_id is null


In [ ]:
dbt("build --select tag:gold tests/")

`assert_every_order_has_lines` is configured as a **warning**: some orders lose their only line during cleaning
(it was a dirty line). Discuss: is that acceptable? What would you change?

## 8. Ask the data a question

Does the seasonality show? Ski sells in winter, swimming in summer:

In [ ]:
m = q(f'''
    SELECT month, category, net_revenue
    FROM gold.{SCHEMA}.mart_revenue_by_category_month
    WHERE year(month) = 2024 AND category IN ('Ski & Snowboard', 'Swimming')
''')
m.pivot(index="month", columns="category", values="net_revenue").astype(float).round(0)

In [ ]:
q(f'SELECT * FROM gold.{SCHEMA}.mart_sales_person_ranking ORDER BY revenue_rank LIMIT 10')

The ranking mart also shows the **cumulative revenue share**: how many sales persons make 80% of the revenue?

## Recap

* Star schema: facts (events, measures) and dimensions (descriptions).
* Jinja `{{ }}` / `{% %}`; macros are reusable functions; packages bring more.
* `dbt build --select tag:gold` builds and tests a whole layer.

---

In [ ]:
# restore_checkpoint(5)